# Testing litellm balancing across 2 vLLM
Document the process of deploying 2 instances of the same model with vLLM from HuggingFace and local balance with litellm. Test the basic model routing features of Litellm and run some basic performance testing with AIPerf

## Install vLLM and deploy a model
```
mkdir perftest
cd perftest/
uv init
uv add vllm
vllm --version
export HUGGING_FACE_HUB_TOKEN=hf_...
CUDA_VISIBLE_DEVICES=0 vllm serve TinyLlama/TinyLlama-1.1B-Chat-v1.0 --gpu-memory-utilization 0.4 
```
### Deploy the second instance in another terminal
Both are running on the same GPU. By default the first instance takes port 8000. We are pinning explicitly the second instance to port 8001 
```
CUDA_VISIBLE_DEVICES=0 vllm serve TinyLlama/TinyLlama-1.1B-Chat-v1.0 --gpu-memory-utilization 0.4 --port 8001
```


## Verify the models are deployed

In [1]:
!nvidia-smi

Tue Jun 16 07:27:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX Pro 6000 Blac...    On  |   00000000:02:00.0 Off |                    0 |
| N/A   N/A    P0            N/A  /  N/A  |   80943MiB /  98304MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
print("First instance on port 8000")
!curl http://localhost:8000/v1/models
print("\nSecond instance on port 8001")
!curl http://localhost:8001/v1/models

First instance on port 8000
{"object":"list","data":[{"id":"TinyLlama/TinyLlama-1.1B-Chat-v1.0","object":"model","created":1781591388,"owned_by":"vllm","root":"TinyLlama/TinyLlama-1.1B-Chat-v1.0","parent":null,"max_model_len":2048,"permission":[{"id":"modelperm-973f338861f3b22a","object":"model_permission","created":1781591388,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}
Second instance on port 8001
{"object":"list","data":[{"id":"TinyLlama/TinyLlama-1.1B-Chat-v1.0","object":"model","created":1781591389,"owned_by":"vllm","root":"TinyLlama/TinyLlama-1.1B-Chat-v1.0","parent":null,"max_model_len":2048,"permission":[{"id":"modelperm-8464e8f8e6f4aa8b","object":"model_permission","created":1781591389,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":

Quick completion test

In [ ]:
from openai import OpenAI

# Point the OpenAI client at your local vLLM server (on v1)
client = OpenAI(
        api_key="doesntmatter",
        base_url="http://localhost:8000/v1",
        )

response = client.chat.completions.create(
        model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Say hello and confirm you are running."},
            ],
         temperature=0.7,
         )

print(response.choices[0].message.content)


Sure, I'm running.


## Deploy litellm
First we need to install litellm
```
uv add litellm litellm[proxy]
uv run litellm --version
uv run litellm --help
```

## Create a `config.yaml` that points to two vLLM endpoints

```
odel_list:
  - model_name: shared-model
    litellm_params:
      model: openai/TinyLlama/TinyLlama-1.1B-Chat-v1.0
      api_base: http://localhost:8000/v1
      api_key: dummy
      #tpm: 2000 # Tokens per minute
      #rpm: 2    # Requests per minute
      #order: 2  # order takes precedence over weight if present
      #weight: 2 # Serve 2 requests from this model for each of the other model

  - model_name: shared-model
    litellm_params:
      model: openai/TinyLlama/TinyLlama-1.1B-Chat-v1.0
      api_base: http://localhost:8001/v1
      api_key: dummy
      #tpm: 10000
      #rpm: 5
      #order: 1
      #weight: 1

router_settings:
  routing_strategy: simple-shuffle
```
**Other advanced features like keys require a PostgreSQL database and are configured in the GUI at "/ui"**

## Let's deploy it on port 4000
```
uv run litellm --config config.yaml --port 4000
```

In [7]:
from openai import OpenAI

# Point the OpenAI client at your local vLLM server (on v1)
client = OpenAI(
        api_key="doesntmatter",
        base_url="http://localhost:4000/v1",
        )

response = client.chat.completions.create(
        model="shared-model",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Say hello and confirm you are running."},
            ],
         temperature=0.7,
         )

print(response.choices[0].message.content)

Sure, I'm running.


## Install AIPerf
```
mkdir perftool
cd perftool/
uv init
uv add aiperf
uv pip freeze | grep aiperf
source .venv/bin/activate
aiperf --version
```

### Performance test with one model down
I had to install aiperf on its own folder/venv due to unsolvable dependencies

In [ ]:
aiperf profile --model 'shared-model' --streaming --endpoint-type 'chat' --tokenizer 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' --url 
'http://localhost:4000/v1' --isl 100 --osl 100 --request-count 200 --concurrency 2

```
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃                                        Metric ┃    avg ┃    min ┃    max ┃    p99 ┃    p90 ┃    p50 ┃   std ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│                      Time to First Token (ms) │  21.25 │  15.28 │  74.24 │  62.06 │  22.94 │  19.29 │  8.64 │
│                     Time to Second Token (ms) │   0.77 │   0.00 │   4.44 │   3.32 │   1.24 │   0.76 │  0.63 │
│               Time to First Output Token (ms) │  21.25 │  15.28 │  74.24 │  62.06 │  22.94 │  19.29 │  8.64 │
│                          Request Latency (ms) │ 258.90 │ 147.10 │ 311.51 │ 294.61 │ 262.04 │ 259.00 │ 13.11 │
│                      Inter Token Latency (ms) │   2.41 │   2.11 │   2.48 │   2.48 │   2.44 │   2.42 │  0.03 │
│              Output Token Throughput Per User │ 414.45 │ 402.86 │ 473.68 │ 438.93 │ 417.63 │ 413.91 │  6.30 │
│                             (tokens/sec/user) │        │        │        │        │        │        │       │
│ E2E Output Token Throughput (tokens/sec/user) │ 384.52 │ 321.02 │ 443.13 │ 395.49 │ 391.04 │ 385.91 │ 11.50 │
│               Output Sequence Length (tokens) │  99.47 │  56.00 │ 103.00 │ 101.00 │ 100.00 │ 100.00 │  4.03 │
│                Input Sequence Length (tokens) │ 100.09 │  99.00 │ 101.00 │ 101.00 │ 100.10 │ 100.00 │  0.31 │
│          Output Token Throughput (tokens/sec) │ 759.90 │    N/A │    N/A │    N/A │    N/A │    N/A │   N/A │
│             Request Throughput (requests/sec) │   7.64 │    N/A │    N/A │    N/A │    N/A │    N/A │   N/A │
│                      Request Count (requests) │ 200.00 │    N/A │    N/A │    N/A │    N/A │    N/A │   N/A │
└───────────────────────────────────────────────┴────────┴────────┴────────┴────────┴────────┴────────┴───────┘

```

### Same AIPerf test with both models running

In [ ]:
aiperf profile --model 'shared-model' --streaming --endpoint-type 'chat' --tokenizer 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' --url 
'http://localhost:4000/v1' --isl 100 --osl 100 --request-count 200 --concurrency 2

```
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┓
┃                                        Metric ┃    avg ┃    min ┃    max ┃    p99 ┃    p90 ┃    p50 ┃    std ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━┩
│                      Time to First Token (ms) │  19.02 │  15.36 │  56.59 │  27.89 │  21.03 │  18.48 │   3.59 │
│                     Time to Second Token (ms) │   2.01 │   0.00 │   5.91 │   4.39 │   3.66 │   1.51 │   1.38 │
│               Time to First Output Token (ms) │  19.02 │  15.36 │  56.59 │  27.89 │  21.03 │  18.48 │   3.59 │
│                          Request Latency (ms) │ 356.42 │  82.16 │ 524.16 │ 488.61 │ 482.88 │ 348.01 │ 103.99 │
│                      Inter Token Latency (ms) │   3.47 │   2.26 │   5.17 │   4.79 │   4.69 │   3.53 │   1.00 │
│              Output Token Throughput Per User │ 314.83 │ 193.29 │ 442.03 │ 430.53 │ 423.74 │ 283.30 │  92.07 │
│                             (tokens/sec/user) │        │        │        │        │        │        │        │
│ E2E Output Token Throughput (tokens/sec/user) │ 297.91 │ 182.57 │ 404.08 │ 400.80 │ 396.40 │ 272.15 │  82.42 │
│               Output Sequence Length (tokens) │  98.31 │  15.00 │ 102.00 │ 101.00 │ 100.00 │ 100.00 │   9.33 │
│                Input Sequence Length (tokens) │ 100.09 │ 100.00 │ 101.00 │ 101.00 │ 100.00 │ 100.00 │   0.29 │
│          Output Token Throughput (tokens/sec) │ 547.88 │    N/A │    N/A │    N/A │    N/A │    N/A │    N/A │
│             Request Throughput (requests/sec) │   5.57 │    N/A │    N/A │    N/A │    N/A │    N/A │    N/A │
│                      Request Count (requests) │ 200.00 │    N/A │    N/A │    N/A │    N/A │    N/A │    N/A │
└───────────────────────────────────────────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┘
```

These setup with 2 models running on the same card produces less throughput